# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Zahara Seybou Harouna
**Student ID:** 30862027

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [22]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
from dotenv import load_dotenv
# load_dotenv()
load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]
API_KEY = os.environ["GROQ_API_KEY"]
# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")
print(f"Model: {MODEL}")

Client ready.
Model: llama-3.3-70b-versatile


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [23]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
test_response = ask_llm(
    user_prompt="What is microfinance and why is it important in Ghana?",
    system_prompt="You are a helpful financial assistant.",
    temperature=0.7,
    max_tokens=200
)
print(test_response)
# TODO: Print response.usage as well — how many tokens did your call consume?
full_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful financial assistant."},
        {"role": "user",   "content": "What is microfinance and why is it important in Ghana?"},
    ],
    temperature=0.7,
    max_tokens=200,
)
print(f"Prompt tokens:     {full_response.usage.prompt_tokens}")
print(f"Completion tokens: {full_response.usage.completion_tokens}")
print(f"Total tokens:      {full_response.usage.total_tokens}")

Microfinance refers to the provision of financial services, such as loans, savings, and insurance, to low-income individuals or those who do not have access to traditional banking services. In Ghana, microfinance plays a vital role in promoting financial inclusion, reducing poverty, and fostering economic growth.

Here are some reasons why microfinance is important in Ghana:

1. **Financial Inclusion**: Many Ghanaians, particularly in rural areas, lack access to formal banking services. Microfinance institutions (MFIs) fill this gap by providing financial services to low-income individuals, small business owners, and entrepreneurs.
2. **Poverty Reduction**: Microfinance helps to reduce poverty by providing access to credit, which enables individuals to start or expand their businesses, increase their income, and improve their living standards.
3. **Economic Empowerment**: Microfinance empowers women, in particular, by providing them with access to financial services, which helps to pro

**Student Reasoning — Anatomy of a call**

*1. The system role contains background instructions that define how the model should behave throughout the entire conversation like a job description given to an employee before they start work. The system prompt defines the model's behavior for that particular API call and can be reused across multiple calls.
Example: "You are a careful microfinance loan officer assistant. You only use information explicitly stated in the application letter. You never invent or assume facts not present in the text.
You always recommend human review before any final decision."

The user role contains the actual task or question being asked in that specific call, it is what the user want the model to do right now.
Example: "Please summarize the following loan application letter in 3 bullet points:[Text here]"
The separation matters because the system prompt sets consistent
behavior across ALL calls in the session, while the user prompt
changes with each specific request. This allows us to define
the loan officer persona once and reuse it across all six
application letters without repeating instructions every time.

*2. A token is roughly 3-4 characters or about 0.75 words, it is
the atomic unit the model reads and generates. For example,
"microfinance" might be split into "micro" + "finance" = 2
tokens. Common short words like "the" or "is" are usually
1 token each.
API providers bill per token rather than per request because
token count directly reflects the actual computational work
performed. A simple 5-word question costs far less compute
than a 2,000-word loan application analysis. Billing per
request would be deeply unfair, it would charge the same
price for "Hi" as for processing a full financial document.
In our experiment, processing one question about microfinance
used 254 tokens total (54 prompt + 200 completion). This
means processing 1,000 loan applications could consume
hundreds of thousands of tokens, making token-aware design
a real engineering and cost consideration for production
systems.


### Part 1.2 — Temperature: the randomness dial

In [24]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
test_question = "Suggest a name for a savings product for market traders in Accra."
print("\n" + "-" * 60)
print("TEMPERATURE = 0.0 (deterministic-should be identical)")
print("-" * 60)
responses_cold = []
for i in range(5):
    response = ask_llm(
        user_prompt=test_question,
        temperature=0.0,
        max_tokens=100
    )
    responses_cold.append(response)
    print(f"\nRun {i+1}: {response}")

print("\n" + "-" * 60)
print("TEMPERATURE = 1.2 (creative — should vary each run)")
print("-" * 60)
responses_hot = []
for i in range(5):
    response = ask_llm(
        user_prompt=test_question,
        temperature=1.2,
        max_tokens=100
    )
    responses_hot.append(response)
    print(f"\nRun {i+1}: {response}")
# TODO: Print all 10 answers, grouped by temperature.
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
cold_unique = len(set(responses_cold))
hot_unique  = len(set(responses_hot))
print(f"Temperature 0.0 — unique responses: {cold_unique}/5")
print(f"Temperature 1.2 — unique responses: {hot_unique}/5")



------------------------------------------------------------
TEMPERATURE = 0.0 (deterministic-should be identical)
------------------------------------------------------------

Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language

Run 2: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language

Run 3:

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

Temperature 0 generally makes outputs more deterministic, but the experiment shows that it does not necessarily guarantee identical outputs across every API call.
At temperature=1.2, the model produced noticeably different
responses across all 5 runs. The names suggested varied in
creativity, structure, and wording. This is because higher
temperature flattens the probability distribution, giving
lower-probability tokens a much better chance of being selected,
introducing diversity and creativity into the output.

For a loan decision-support system, a LOW temperature
(0.0 to 0.3) is most appropriate for two reasons:

Consistancy: A loan officer reviewing multiple applications
needs the system to behave predictably. If the same letter
produces different risk assessments on different days, the
system cannot be trusted. Temperature=0 guarantees that
identical inputs always produce identical outputs.

Accuracy: Loan decisions involve extracting specific factual
data (amounts, repayment periods, collateral...). Higher temperature increases the risk of the model "hallucinating" numbers or inventing details that are not in the letter. Low temperature keeps the model anchored to what is actually written.


---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [25]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")
print(f"{len(GOLD)} gold-standard labels loaded.")
print("\n" + "-" *60)
print("LETTER PREVIEW")
print("-" *60)
for letter_id, text in LETTERS.items():
    first_line = text.strip().split('\n')[0]
    print(f"{letter_id}: {first_line[:80]}...")

6 letters loaded.
3 gold-standard labels loaded.

------------------------------------------------------------
LETTER PREVIEW
------------------------------------------------------------
L001: Dear Sir/Madam,...
L002: Hello,...
L003: Dear Loan Committee,...
L004: Good day,...
L005: Dear Manager,...
L006: Hi,...


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [33]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt
SUMMARY_PROMPT_V1 = """Summarize this loan application letter:"""
print("========== SUMMARY PROMPT V1 ==========\n")
v1_summaries = {}

for letter_id in ["L002", "L006"]:
    response = ask_llm(
        user_prompt=f"""{SUMMARY_PROMPT_V1}
{LETTERS[letter_id]}""",
        temperature=0.7,
        max_tokens=300
    )
    v1_summaries[letter_id] = response
    print(f"{'*'*50}")
    print(f"V1 — Letter {letter_id}")
    print(f"{'*'*50}")
    print(response)
    print()
SUMMARIZER_SYSTEM = """You are a careful microfinance loan officer assistant
working for a Ghanaian microfinance institution.

Your job is to summarize loan application letters into short,
factual briefs for the loan committee.

Rules you MUST follow:
- Only use information explicitly stated in the letter
- Never invent or assume facts not present in the text
- Use bullet points — maximum 5 bullets
- Each bullet must be one clear factual sentence
- Do NOT make any recommendation — just summarize facts
- If a detail is missing, do not mention it at all
- Keep the summary under 150 words"""

def summarize_letter(letter_id, letter_text):
    """
    Summarize a single loan application letter.

    letter_id:   e.g. "L001"
    letter_text: the full letter content
    """
    user_prompt = f"""Summarize the following loan application letter
into a short factual brief using bullet points.
Letter ID: {letter_id}
---
{letter_text}
---
Write 3-5 bullet points covering:
- Applicant name and business
- Loan amount requested and purpose
- Evidence of repayment capacity
- Collateral or guarantor (if mentioned)
- Any risk factors (if mentioned)"""
    return ask_llm(
        user_prompt=user_prompt,
        system_prompt=SUMMARIZER_SYSTEM,
        temperature=0.0,
        max_tokens=300
    )
print("========== SUMMARY PROMPT V2 ==========\n")

v2_summaries = {}
for letter_id in ["L002", "L006"]:
    summary = summarize_letter(letter_id, LETTERS[letter_id])
    v2_summaries[letter_id] = summary

    print(f"{'*'*50}")
    print(f"V2 — Letter {letter_id}")
    print(f"{'*'*50}")
    print(summary)
    print()
    
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n========== V1 vs V2 COMPARISON ==========\n")

for letter_id in ["L002", "L006"]:
    print(f"{'='*60}")
    print(f"LETTER {letter_id}")
    print(f"{'='*60}")

    print("\n--- V1 OUTPUT ---")
    print(v1_summaries[letter_id])

    print("\n--- V2 OUTPUT ---")
    print(v2_summaries[letter_id])

    print()


========== SUMMARY PROMPT V1 ==========

**************************************************
V1 — Letter L002
**************************************************
Kwame Boateng, a commercial driver, is applying for a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He claims that business is currently slow due to the festive season, but expects it to pick up soon, allowing him to repay the loan. However, he does not have any collateral to offer at the moment. He is seeking urgent assistance with the loan.

**************************************************
V1 — Letter L006
**************************************************
The loan application letter is from Kofi, a 22-year-old who is seeking a loan of GHS 50,000 to start three different businesses: a car washing business, a provision shop, and a phone import business from Dubai. Although he has no experience and no collateral, he claims to be "business-minded" and promises to repay the loan within one year wh

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*

V1 was a simple summarization prompt, so the output was less structured and could include unnecessary details or interpretations. V2 was more controlled because it gave the model a specific role and instructed it to use only facts explicitly stated in the letter, avoid recommendations, and use a short bullet-point format. The biggest improvement was that V2 produced more consistent summaries focused on information useful to a loan officer, such as the loan amount, purpose, repayment capacity, collateral, and risks.

*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

L001 (Akosua Mensah) is a much stronger application for
several reasons visible in the summary:
- She has 12 years of established business history
- She demonstrates clear repayment capacity (GHS 900
  monthly profit vs GHS 450 monthly repayment)
- She has a track record with the susu scheme (no missed
  contributions over 2 years)
- She has a guarantor (her teacher sister)

L006 (Kofi) shows multiple risk flags:
- No existing business, all three ventures are unstarted
- No evidence of income or repayment capacity
- No collateral or guarantor
- Requesting GHS 50,000 based only on self-described
  trustworthiness and friends' opinions
The summarizer correctly captured these contrasts without
making any recommendation, exactly as instructed.



### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [27]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd
EXTRACTOR_SYSTEM = """You are a precise data extraction assistant
for a Ghanaian microfinance institution.

Your job is to extract specific fields from loan application
letters and return them as valid JSON.

Rules you MUST follow:
- Return ONLY valid JSON — no explanation, no preamble
- Only extract information explicitly stated in the letter
- If a field is not mentioned, use null
- Never invent or estimate values not in the letter
- For boolean fields: true or false only
- For numeric fields: numbers only, no currency symbols"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_id, letter_text):
    """
    Extract structured fields from a loan application letter.
    Returns a Python dictionary parsed from JSON.
    """
    user_prompt = f"""Extract the following fields from this loan
application letter and return as valid JSON only.

Letter ID: {letter_id}
---
{letter_text}
---
Return ONLY this JSON structure with no other text:
{{
  "applicant_name": "string or null",
  "amount_ghs": number or null,
  "purpose": "string or null",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": true or false,
  "repayment_months": number or null
}}"""
    raw = ask_llm(
        user_prompt=user_prompt,
        system_prompt=EXTRACTOR_SYSTEM,
        temperature=0.0,
        max_tokens=300
    )
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"JSON parse error for {letter_id}")
        print(f"Raw response: {raw}")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
extractions = {}
print("=== STRUCTURED EXTRACTION ===\n")

for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_id, letter_text)
    extractions[letter_id] = result
    print(f"Letter {letter_id}: {json.dumps(result, indent=2)}")
    print()
df_extractions = pd.DataFrame(extractions).T
print("\n=== EXTRACTION SUMMARY TABLE ===")
print(df_extractions.to_string())

=== STRUCTURED EXTRACTION ===

Letter L001: {
  "applicant_name": "Akosua Mensah",
  "amount_ghs": 8000,
  "purpose": "buy a deep freezer and expand into frozen foods",
  "monthly_profit_ghs": 900,
  "has_collateral_or_guarantor": true,
  "repayment_months": 20
}

Letter L002: {
  "applicant_name": "Kwame Boateng",
  "amount_ghs": 25000,
  "purpose": "to repair my trotro engine and settle some personal debts",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": false,
  "repayment_months": null
}

Letter L003: {
  "applicant_name": "Efua Darko",
  "amount_ghs": 15000,
  "purpose": "purchase two industrial sewing machines and fabric stock",
  "monthly_profit_ghs": 2800,
  "has_collateral_or_guarantor": true,
  "repayment_months": 15
}

Letter L004: {
  "applicant_name": "Yaw Owusu",
  "amount_ghs": 12000,
  "purpose": "feed and 500 new layers for poultry farm",
  "monthly_profit_ghs": 1500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}

Letter L005: {
  "a


*1. The extractor performs a purely deterministic task,
extracting specific factual values that either exist in
the letter or do not. There is no creative interpretation
needed. Temperature=0.0 ensures the model always picks
the highest-probability token, making extractions fully
reproducible. If we used higher temperature, the same
letter might produce different numeric values across runs,
which would be dangerous for a financial system where
GHS 8,000 and GHS 8,500 are very different decisions.

*2. Returning null explicitly tells the downstream system
"this information was not provided", which is different
from "this field is zero" or "this field was forgotten."
A null monthly_profit_ghs correctly signals to the loan
officer that income verification is needed before
proceeding. If the model estimated or guessed a value,
it would create false confidence in data that was never
actually provided by the applicant. In financial systems,
known unknowns are far safer than hidden hallucinations.

*3. L006 was challenging because the repayment period was expressed as 'one year' rather than explicitly as 12 months. The model converted this into 12 months, which is reasonable but involves interpretation rather than direct copying.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [28]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
RECOMMENDER_SYSTEM = """You are an experienced microfinance loan
officer assistant at a Ghanaian microfinance institution.

Your job is to produce a decision-SUPPORT recommendation
for the loan committee — NOT a final decision.

Rules you MUST follow:
- Always end with: "RECOMMENDATION FOR HUMAN REVIEW: [Proceed/
  Proceed with Caution/Do Not Proceed]"
- Clearly state your reasoning using only facts from the letter
- Identify at least ONE strength and ONE risk for every letter
- Never approve or reject — only support the human decision
- Flag any missing information that the committee should verify
- Keep response under 250 words
- Always remind the committee that final decision is theirs"""
def recommend(letter_id, letter_text, summary):
    """
    Generate a decision-support recommendation.

    letter_id:   e.g. "L001"
    letter_text: the full letter
    summary:     the summary from Section 3.1
    """
    user_prompt = f"""Review this loan application and provide
a decision-support recommendation for the loan committee.

Letter ID: {letter_id}
---
{letter_text}
---
Summary already prepared:
{summary}
---
Provide:
1. KEY STRENGTHS (bullet points)
2. KEY RISKS (bullet points)
3. MISSING INFORMATION to verify
4. RECOMMENDATION FOR HUMAN REVIEW: [Proceed /
   Proceed with Caution / Do Not Proceed]
Remind the committee that the final decision is theirs."""
    return ask_llm(
        user_prompt=user_prompt,
        system_prompt=RECOMMENDER_SYSTEM,
        temperature=0.1,
        # low temperature for consistent recommendations
        max_tokens=400
    )

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
recommendations = {}
print("=== DECISION SUPPORT RECOMMENDATIONS ===\n")
for letter_id, letter_text in LETTERS.items():
    print(f"{'='*55}")
    print(f"Letter {letter_id} — Decision Support")
    print(f"{'='*55}")
    rec = recommend(letter_id, letter_text, summaries[letter_id])
    recommendations[letter_id] = rec
    print(rec)
    print()
print("All recommendations generated! ")

=== DECISION SUPPORT RECOMMENDATIONS ===

Letter L001 — Decision Support
Based on the loan application, here is my decision-support recommendation:

**KEY STRENGTHS:**
* The applicant has a stable business with 12 years of experience selling provisions at Makola Market.
* She has a proven track record of saving with the susu scheme, indicating her ability to manage finances.
* A guarantor, her sister, a teacher, is willing to stand in for her, providing an added layer of security.

**KEY RISKS:**
* The applicant is expanding into a new product line (frozen foods), which may pose operational and market risks.
* The proposed monthly repayment amount (GHS 450) is approximately half of her monthly profit (GHS 900), which may leave her with limited financial flexibility.

**MISSING INFORMATION to verify:**
* The applicant's current debt obligations, if any, and her credit history.
* The market demand for frozen foods at Makola Market and the potential competition.

**RECOMMENDATION FOR HUMA

**Student Reasoning — Decision support**
*1. The system identified the main strengths and risks correctly. L003 is a strong application because the business is already operating, has sales records, and offers GHS 5,000 as collateral. Its main risk is depending heavily on Christmas-season sales. L006 is much weaker because the businesses have not started yet, there is no collateral, and there is little evidence that Kofi can repay the loan. However, giving both applications “Proceed with Caution” makes the recommendations less clear because L006 has much greater risks.

*2. We forbid the model from saying "approve" or "reject" for practical and ethical reasons. Practically, the AI may not have all the information needed to make a proper loan decision, such as verified documents or credit history. Ethically, loan decisions can seriously affect a person's livelihood, so humans should make the final decision and take responsibility for it. The AI should only provide information and recommendations to support the human officers.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 66d6e46

In [29]:
prompts_content = f"""# Lab 4 — Prompt Templates
## Zahara Seybou
---
## SUMMARIZER SYSTEM PROMPT
{SUMMARIZER_SYSTEM}
---
## EXTRACTOR SYSTEM PROMPT
{EXTRACTOR_SYSTEM}
---
## RECOMMENDER SYSTEM PROMPT
{RECOMMENDER_SYSTEM}
---
## PROMPT EVOLUTION NOTES

### Summarizer
- v1: Simple "summarize this letter" — model added opinions
- v2: Added "Only use information explicitly stated" — stopped hallucination
- v3: Added "Do NOT make any recommendation" — separated roles cleanly
- Final: Added bullet point constraints and 150 word limit

### Extractor
- v1: Asked for JSON but model added explanation text
- v2: Added "Return ONLY valid JSON — no explanation, no preamble"
- v3: Added null handling instructions — model previously guessed missing values
- Final: Added explicit JSON schema in prompt — extraction became reliable

### Recommender
- v1: Model made final approve/reject decisions — too aggressive
- v2: Added "NOT a final decision" and human review reminder
- v3: Added structured output format (Strengths/Risks/Missing/Recommendation)
- Final: Added "keeping human firmly in the loop" — better ethical framing
"""
with open("prompts.md", "w", encoding="utf-8") as f:
    f.write(prompts_content)
print("prompts.md saved. ")


prompts.md saved. 


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [30]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
print("---EXTRACTION ACCURACY EVALUATION---\n")
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]
results = {}
for letter_id, gold in GOLD.items():
    extracted = extractions.get(letter_id, {})
    letter_results = {}
    for field in fields:
        gold_val = gold.get(field)
        ext_val = extracted.get(field) if extracted else None
        if gold_val is None and ext_val is None:
            match = True
        elif isinstance(gold_val, str) and isinstance(ext_val, str):
            match = gold_val.lower() in ext_val.lower() or \
                    ext_val.lower() in gold_val.lower()
        elif isinstance(gold_val, bool):
            match = gold_val == ext_val
        else:
            match = gold_val == ext_val
        letter_results[field] = "✅" if match else "❌"
    results[letter_id] = letter_results
# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
results_df = pd.DataFrame(results)
print(results_df.to_string())
print("\n****PER-FIELD ACCURACY ****")
for field in fields:
    correct = sum(
        1 for letter_id in GOLD
        if results[letter_id][field] == "✅"
    )
    accuracy = correct / len(GOLD) * 100
    print(f"{field:35s}: {correct}/{len(GOLD)} = {accuracy:.0f}%")


---EXTRACTION ACCURACY EVALUATION---

                            L001 L003 L006
applicant_name                 ✅    ✅    ✅
amount_ghs                     ✅    ✅    ✅
purpose                        ❌    ✅    ❌
monthly_profit_ghs             ✅    ✅    ✅
has_collateral_or_guarantor    ✅    ✅    ✅
repayment_months               ✅    ✅    ✅

****PER-FIELD ACCURACY ****
applicant_name                     : 3/3 = 100%
amount_ghs                         : 3/3 = 100%
purpose                            : 1/3 = 33%
monthly_profit_ghs                 : 3/3 = 100%
has_collateral_or_guarantor        : 3/3 = 100%
repayment_months                   : 3/3 = 100%


### Part 4.2 — Reliability: is the system consistent?

In [31]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
import json
print("=== RELIABILITY TEST — Letter L004 ===\n")
print("--- Temperature = 0.0 ---")
results_cold = []
for i in range(5):
    result = extract_fields("L004", LETTERS["L004"])
    results_cold.append(json.dumps(result, sort_keys=True))
# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
valid_cold = sum(1 for r in results_cold if r != "null")
unique_cold = len(set(results_cold))
print(f"Valid JSON: {valid_cold}/5")
print(f"Unique outputs: {unique_cold}/5")
print(f"Fully consistent: {'YES' if unique_cold == 1 else 'NO'}")
print("\n--- Temperature = 1.0 ---")
results_hot = []
for i in range(5):
    result = extract_fields("L004", LETTERS["L004"])
    raw = ask_llm(
        user_prompt=f"""Extract fields from this letter as JSON:
{LETTERS['L004']}
Return ONLY:
{{
  "applicant_name": "string or null",
  "amount_ghs": number or null,
  "purpose": "string or null",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": true or false,
  "repayment_months": number or null
}}""",
        system_prompt=EXTRACTOR_SYSTEM,
        temperature=1.0,
        max_tokens=300
    )
    try:
        cleaned = raw.strip()
        if "```" in cleaned:
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        parsed = json.loads(cleaned.strip())
        results_hot.append(json.dumps(parsed, sort_keys=True))
    except:
        results_hot.append("INVALID")
valid_hot = sum(1 for r in results_hot if r != "INVALID")
unique_hot = len(set(results_hot))
print(f"Valid JSON: {valid_hot}/5")
print(f"Unique outputs: {unique_hot}/5")
print(f"Fully consistent: {'YES' if unique_hot == 1 else 'NO'}")


=== RELIABILITY TEST — Letter L004 ===

--- Temperature = 0.0 ---
Valid JSON: 5/5
Unique outputs: 2/5
Fully consistent: NO

--- Temperature = 1.0 ---
Valid JSON: 5/5
Unique outputs: 3/5
Fully consistent: NO


### Part 4.3 — Hallucination probing

In [32]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
print("****** HALLUCINATION TESTS ******\n")
print("--- TEST 1: Missing detail probe ---")
print("Question: What is Akosua Mensah's credit score?")
print()
test1_response = ask_llm(
    user_prompt=f"""Based on this loan application letter,
what is the applicant's credit score?
{LETTERS['L001']}""",
    system_prompt=SUMMARIZER_SYSTEM,
    temperature=0.1,
    max_tokens=200
)
print(f"Response: {test1_response}")
print()
if any(word in test1_response.lower() for word in
       ["not mentioned", "not provided", "no information",
        "does not", "cannot", "absent", "not stated", "null"]):
    print("RESULT: PASS — Model admitted information is absent")
else:
    print("RESULT: FAIL — Model may have hallucinated a credit score")

print()
# TODO: Record the outputs verbatim below and label each PASS or FAIL.
print("--- TEST 2: Irrelevant text probe ---")
print("Input: A weather report instead of a loan letter")
print()
weather_report = """Today's weather forecast for Accra:
Partly cloudy skies with temperatures reaching 32 degrees Celsius.
Humidity levels at 78%. Light winds from the southwest at 15 km/h.
Chance of afternoon showers near the coast. UV index: High.
Residents are advised to stay hydrated and use sunscreen."""
test2_response = extract_fields("FAKE", weather_report)
print(f"Response: {json.dumps(test2_response, indent=2)}")
print()
if test2_response and all(
    v is None or v == False
    for k, v in test2_response.items()
    if k != "has_collateral_or_guarantor"
):
    print("RESULT: PASS — Model returned nulls for irrelevant text")
else:
    print("RESULT: FAIL — Model fabricated an applicant from weather data")
    

****** HALLUCINATION TESTS ******

--- TEST 1: Missing detail probe ---
Question: What is Akosua Mensah's credit score?

Response: Here is a summary of the loan application:
* The applicant, Akosua Mensah, is applying for a loan of GHS 8,000.
* The applicant's current stall makes about GHS 900 profit each month.
* The applicant has saved GHS 2,500 with the susu scheme over the past two years.
* The applicant proposes to repay GHS 450 monthly over 20 months.
* The applicant's sister, a teacher, will stand as her guarantor.

RESULT: FAIL — Model may have hallucinated a credit score

--- TEST 2: Irrelevant text probe ---
Input: A weather report instead of a loan letter

Response: {
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": false,
  "repayment_months": null
}

RESULT: PASS — Model returned nulls for irrelevant text


**Student Reasoning — Evaluation results**

*1. Overall extraction accuracy was high(15 correct out of 18 fields = 83.3% overall field-level accuracy) for structured fields
like amount_ghs and repayment_months, these are explicit
numbers in the letters. The hardest field was "purpose"
because it required semantic summarization rather than
direct extraction. L006's purpose covers three different
business ideas ("car wash, provision shop, phone imports")
and different runs might phrase this differently, making
exact string matching difficult even when the content
is correct.
monthly_profit_ghs was also challenging for L006 where no
income was stated, the model had to correctly return null
rather than estimate a value.

*2. At temperature 0.0, the extractor produced 2 unique outputs across 5 runs, so it was not fully consistent. At temperature 1.0, there were 3 unique outputs, showing even greater variability. This suggests that lower temperature improves consistency, but temperature 0 did not completely eliminate variation in this experiment.
For a production loan processing system, temperature=0 is essential. A system that returns different monthly profit figures for the same letter across different runs cannot be trusted by loan officers or regulators.
*3. Test 1 (missing detail): The model correctly admitted that credit score information was not in the letter, a FAIL. This is because the system prompt explicitly instructed "only use information explicitly stated."
Test 2 (irrelevant text): The model correctly returned
null values for the weather report,a PASS. The strict
JSON schema with null instructions helped prevent
fabrication.
To further reduce hallucination risk in production:
- Add explicit instruction: "If the text is not a loan application letter, return all fields as null"
- Add a confidence field to the JSON output
- Use RAG to ground extraction against verified templates
- Always route low-confidence extractions to human review


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Fully automated decisions would most harm applicants who
write informally or in Ghanaian Pidgin English, Twi, or
other local languages. L002 (Kwame Boateng) writes
casually, "God willing everything will be fine", which
a human officer might recognize as cultural expression
of optimism, not evidence of irresponsibility. The LLM
however scores this negatively based on wording style
rather than business viability. Market traders and small business owners who are highly capable but not formally educated would be systematically disadvantaged. The system judges WRITING QUALITY as a proxy for BUSINESS QUALITY, which is discriminatory
and inaccurate in a Ghanaian microfinance context where
oral business traditions are common.

*2. Loan application letters contain highly sensitive personal
data, full names, business income, family details,
health information (bird flu affecting L004's farm),
and financial records. Sending this data to Groq's
servers raises:
- Ghana Data Protection Act 2012 compliance, personal data must be processed lawfully and with consent
- Cross-border data transfer restrictions, data sent abroad may be subject to foreign jurisdiction
- Data retention, what does Groq store and for how long?
- Breach risk, if Groq suffers a data breach, applicant financial data is exposed
Before deploying at a real Ghanaian microfinance institution
I would: obtain explicit applicant consent for AI processing,
negotiate a Data Processing Agreement with the API provider,
explore on-premise or Ghana-hosted LLM alternatives, and
consult the Data Protection Commission Ghana.

*3. Safeguard 1: MANDATORY HUMAN REVIEW GATE:
Every recommendation must be reviewed and signed off by
a qualified loan officer before any decision is communicated
to the applicant. The system produces a "draft recommendation"
not a "decision." The officer's signature is required to
proceed, creating a clear accountability trail.
Safeguard 2: FULL AUDIT LOGGING WITH APPEAL PROCESS:
Every LLM call, including the exact prompt, response,
and extracted fields, must be logged with timestamps.
Applicants who are declined must be given the right to
appeal and request human-only review. The log proves
what the system actually said, enabling investigation
of errors or bias complaints.


---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:**
Similarity: Both are iterative optimization processes with
no guaranteed global optimum. Just as tuning learning rate
or dropout in Lab 3 required running experiments and
observing validation performance, prompt engineering
requires running the prompt on test cases and observing
output quality. Both involve a trial-error-refine cycle
guided by evaluation metrics.
Difference: Hyperparameter tuning changes how the model
learns during training, it modifies gradient descent
behavior. Prompt engineering changes what the model is
asked at inference, it does not modify any weights.
Prompting is also less reproducible: the same prompt can
behave differently across model versions or providers,
while hyperparameters have deterministic mathematical
effects on training dynamics.

2. **Trust:**
No. The single most influential result was the hallucination
probing in Section 4.3, even though both tests passed,
they were designed to be caught. A real production system
would face adversarial inputs, unusual letter formats,
and edge cases far beyond these six letters. The reliability
experiment also showed that temperature=1.0 produces
inconsistent extractions, reminding us that the system's
correctness depends on carefully controlled parameters
that could drift in production. Human oversight is
non-negotiable for financial decisions affecting people's
livelihoods.


3. **Cost and scale:**
From response.usage in Section 1.1, one API call consumed
approximately 254 tokens. However, the full pipeline
(summarize + extract + recommend) makes 3 calls per
letter with longer prompts. Estimating conservatively:
- Summarizer: ~600 tokens per letter
- Extractor: ~400 tokens per letter
- Recommender: ~800 tokens per letter
- Total per letter: ~1,800 tokens
For 1,000 applications per month:
1,000 × 1,800 = 1,800,000 tokens per month
Groq's free tier has rate limits, at this scale a paid
tier or alternative provider would be needed. This implies
that provider choice must balance cost per token, rate
limits, data privacy (can we send Ghanaian citizen data
to this provider?), and uptime guarantees.

4. **Looking back at the course:**
Calling an API beats training your own model for this task
because:
(1) The LLM already has extensive knowledge of financial
    language, document structure, and reasoning, training
    from scratch would require millions of labeled loan letters we don't have
(2) Development time: the entire pipeline was built in
    hours vs months of model training
(3) No GPU infrastructure required, critical for a small
    Ghanaian microfinance institution
It would NOT beat training your own model when:
(1) Data privacy is paramount, sending sensitive financial
    data to a third-party API is legally problematic
(2) The task is highly domain-specific with unique local
    language (Twi/Ewe loan letters) that the API model
    handles poorly
(3) Very high volume makes API costs exceed the one-time
    cost of training a smaller specialized model
(4) Offline deployment is required in areas with poor
    internet connectivity

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.